# Panel Builder – S&P 500 (2015–2025)

Este notebook construye el panel final firm-month utilizado en las estimaciones econométricas.

Input:
- data/raw/ (insumos Refinitiv descargados por ETL)

Output principal:
- data/clean/panel_master.parquet

Outputs auxiliares:
- data/clean/data_dictionary.csv
- data/clean/lineage_log.csv
- outputs/logs/panel_build_*.log
- outputs/qa/qa_report.md

El notebook NO estima modelos.

## 1) Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
from datetime import datetime
from typing import Dict, List

## 2) Configuración y Paths

Input:
- data/raw/

Output:
- data/clean/panel_master.parquet

Notas:
- Todos los paths son relativos al root del proyecto.

In [ ]:
from pathlib import Path

def detect_project_root(markers=("requirements.txt", ".git", "README.md")) -> Path:
    """
    Detecta el root del repo buscando 'markers' hacia arriba desde el cwd.
    Funciona si corrés el notebook desde /notebooks o desde el root.
    """
    here = Path.cwd().resolve()
    for parent in [here] + list(here.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    return here  # fallback

# Root del proyecto (robusto)
PROJECT_ROOT = detect_project_root()

RAW      = PROJECT_ROOT / "data" / "raw"
CLEAN    = PROJECT_ROOT / "data" / "clean"
INTERIM  = PROJECT_ROOT / "data" / "intermediate"

OUTPUTS  = PROJECT_ROOT / "outputs"
LOGS     = OUTPUTS / "logs"
QA       = OUTPUTS / "qa"

# Crear directorios necesarios
for p in [RAW, INTERIM, CLEAN, LOGS, QA]:
    p.mkdir(parents=True, exist_ok=True)

PANEL_PATH = CLEAN / "panel_master.parquet"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW         =", RAW)
print("CLEAN       =", CLEAN)
print("OUTPUTS     =", OUTPUTS)

## 3) Logging

In [ ]:
log_file = LOGS / f"panel_build_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio Panel Builder")

## 4) Carga de insumos

Input:
- spreads bonos
- curvas UST
- equity returns
- fundamentals
- índices crediticios
- mappings sectoriales

Chequeos:
- assert no vacío
- rango temporal esperado

In [ ]:
def load_raw_data() -> Dict[str, pd.DataFrame]:
    data = {}

    data["spreads"] = pd.read_parquet(RAW / "bonds_monthly_spreads.parquet")
    data["equity"] = pd.read_parquet(RAW / "equity_market_returns.parquet")
    data["fundamentals_q"] = pd.read_parquet(RAW / "fundamentals_extended_q.parquet")
    data["credit_indices"] = pd.read_parquet(RAW / "iboxx_credit_indices_monthly.parquet")

    for name, df in data.items():
        assert not df.empty, f"{name} vacío"
        logging.info(f"{name} cargado: {df.shape}")

    return data

raw_data = load_raw_data()

## 5) Spreads por firma

Input:
- bonds_monthly_spreads

Output:
- spread firm-month

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# -------------------------------------------------
# Helpers de IO (parquet/csv)
# -------------------------------------------------
def load_parquet_or_csv(parquet_path: Path, csv_path: Path) -> pd.DataFrame:
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f"No existe ni {parquet_path.name} ni {csv_path.name} en {parquet_path.parent}")

def coerce_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

# -------------------------------------------------
# Inputs (preferimos INTERIM si existe, sino RAW)
# -------------------------------------------------
BONDS_SPREADS_PARQ = (INTERIM / "bonds_monthly_spreads.parquet") if (INTERIM / "bonds_monthly_spreads.parquet").exists() else (RAW / "bonds_monthly_spreads.parquet")
BONDS_SPREADS_CSV  = (INTERIM / "bonds_monthly_spreads.csv")     if (INTERIM / "bonds_monthly_spreads.csv").exists()     else (RAW / "bonds_monthly_spreads.csv")

bonds = load_parquet_or_csv(BONDS_SPREADS_PARQ, BONDS_SPREADS_CSV)
bonds["date"] = pd.to_datetime(bonds["date"], errors="coerce")
bonds = bonds.sort_values(["issuer", "bond_ric", "date"]).copy()

bonds = coerce_numeric(bonds, ["spread_bps", "ytm", "residual_maturity_years"])

print("\n=== bonds (spreads por bono) — preview ===")
print(bonds.head(5))

# -------------------------------------------------
# bonds_meta: usar el que ya cargaste; si no existe, intentar disco
# -------------------------------------------------
if "bonds_meta" not in globals():
    # Ajustá estos nombres si tu ETL los guarda distinto
    META_PARQ = RAW / "bonds_meta.parquet"
    META_CSV  = RAW / "bonds_meta.csv"
    bonds_meta = load_parquet_or_csv(META_PARQ, META_CSV)

# forzar 1 fila por ric en la metadata
bonds_meta = bonds_meta.rename(columns={"ric": "bond_ric"})
bonds_meta_ren = bonds_meta.drop_duplicates(subset=["bond_ric"]).copy()

# -------------------------------------------------
# Merge spreads + metadata
# -------------------------------------------------
bonds_full = bonds.merge(
    bonds_meta_ren,
    on="bond_ric",
    how="left",
    suffixes=("", "_meta")
)

# Normalizar issuer
if "issuer" not in bonds_full.columns:
    issuer_candidates = [c for c in bonds_full.columns if c.lower() == "issuer"]
    if issuer_candidates:
        bonds_full = bonds_full.rename(columns={issuer_candidates[0]: "issuer"})
    else:
        raise KeyError("No encontré columna issuer en bonds_full")

# Normalizar fecha
if "date" not in bonds_full.columns:
    date_candidates = [c for c in bonds_full.columns if c.lower() in ["date", "month"]]
    if date_candidates:
        bonds_full = bonds_full.rename(columns={date_candidates[0]: "date"})
    else:
        raise KeyError("No encontré columna date en bonds_full")

bonds_full["date"] = pd.to_datetime(bonds_full["date"], errors="coerce")

# -------------------------------------------------
# Detectar columna de monto (si existe) – MISMA LÓGICA QUE TU NOTEBOOK
# -------------------------------------------------
amount_candidates = [
    c for c in bonds_full.columns
    if "amount" in c.lower() and ("issued" in c.lower() or "issue" in c.lower())
]
if not amount_candidates:
    amount_candidates = [c for c in bonds_full.columns if "amount" in c.lower()]
amount_col = amount_candidates[0] if amount_candidates else None

# -------------------------------------------------
# Función agregadora issuer-date (MISMA LÓGICA)
# -------------------------------------------------
def agg_func(df: pd.DataFrame) -> pd.Series:
    if amount_col and amount_col in df.columns:
        w = pd.to_numeric(df[amount_col], errors="coerce")
    else:
        w = pd.Series(np.nan, index=df.index)

    out = {}

    # spread promedio
    if w.notna().sum() > 0 and w.sum() > 0:
        out["spread_mean_bps"] = np.average(df["spread_bps"], weights=w)
    else:
        out["spread_mean_bps"] = df["spread_bps"].mean()

    # ytm promedio
    if w.notna().sum() > 0 and w.sum() > 0:
        out["ytm_mean"] = np.average(df["ytm"], weights=w)
    else:
        out["ytm_mean"] = df["ytm"].mean()

    # maturidad residual promedio
    if w.notna().sum() > 0 and w.sum() > 0:
        out["residual_maturity_mean"] = np.average(df["residual_maturity_years"], weights=w)
    else:
        out["residual_maturity_mean"] = df["residual_maturity_years"].mean()

    # contar BONOS DISTINTOS
    out["n_bonds"] = df["bond_ric"].nunique()

    return pd.Series(out)

# -------------------------------------------------
# Agregar a nivel empresa-fecha
# -------------------------------------------------
bonds_monthly_spreads_firm = (
    bonds_full
    .groupby(["issuer", "date"], as_index=False)
    .apply(agg_func)
    .reset_index(drop=True)
    .sort_values(["issuer", "date"])
)

# -------------------------------------------------
# Guardar intermedio (igual que tu notebook)
# -------------------------------------------------
OUT_FIRM_PARQ = INTERIM / "bonds_monthly_spreads_firmlevel.parquet"
OUT_FIRM_CSV  = INTERIM / "bonds_monthly_spreads_firmlevel.csv"

bonds_monthly_spreads_firm.to_parquet(OUT_FIRM_PARQ, index=False)
bonds_monthly_spreads_firm.to_csv(OUT_FIRM_CSV, index=False)

print(f"\n✅ Firm-level spreads guardado:\n{OUT_FIRM_PARQ}\n{OUT_FIRM_CSV}")

### 5.1) QA spreads

In [ ]:
# =====================================================
# QA básica de spreads (sanity check) – MISMA CELDA QUE TENÍAS
# =====================================================
df = bonds.copy()

print("\n--- Estadísticas generales ---")
print(df["spread_bps"].describe())

print("\n--- NAs en spread_bps ---")
print(df["spread_bps"].isna().mean())

print("\n--- Conteo de bonos únicos ---")
print("n bond_ric:", df["bond_ric"].nunique())

print("\n--- Conteo de emisores únicos ---")
print("n issuer:", df["issuer"].nunique())

print("\n--- Rango temporal ---")
print(df["date"].min(), "→", df["date"].max())

print("\n--- Checks por issuer-date (pre-aggregate) ---")
tmp = df.groupby(["issuer", "date"]).size()
print("issuer-date grupos:", tmp.shape[0])
print("max filas por issuer-date:", tmp.max())

## 6. Mapping issuer → ticker → sector

Input:
- Carpeta con excels `bonds_empresas/` (provenientes del proceso de bonos / armado manual)

Output:
- `data/clean/issuer_ticker_sector.parquet`
- `data/clean/issuer_ticker_sector.csv`

Chequeos:
- Se encuentra la carpeta `bonds_empresas/`
- Columnas mínimas detectadas: issuer + (ticker y/o sector)
- Deduplicación por issuer

In [ ]:
from pathlib import Path
import pandas as pd

# -------------------------------------------------
# Localización carpeta de excels
# -------------------------------------------------
cand_0 = PROJECT_ROOT / "data" / "inputs" / "bonds_empresas"
cand_1 = PROJECT_ROOT / "bonds_empresas"
cand_2 = RAW / "bonds_empresas"

BONDS_DIR = cand_1 if cand_1.exists() else cand_2
if not BONDS_DIR.exists():
    raise FileNotFoundError(
        "No encontré la carpeta 'bonds_empresas'. "
        "Colocala en PROJECT_ROOT/bonds_empresas o en data/raw/bonds_empresas."
    )

files = sorted(BONDS_DIR.glob("*.xlsx"))
if len(files) == 0:
    raise FileNotFoundError(f"No hay archivos .xlsx dentro de {BONDS_DIR}")

print(f"📁 bonds_empresas dir: {BONDS_DIR}")
print(f"📄 excels encontrados: {len(files)}")

# -------------------------------------------------
# Lectura de excels (stack)
# -------------------------------------------------
dfs = []
for f in files:
    try:
        df = pd.read_excel(f)
        df["source_file"] = f.name
        dfs.append(df)
    except Exception as e:
        print(f"⚠️ No pude leer {f.name}: {e}")

issuer_map_raw = pd.concat(dfs, ignore_index=True)

# -------------------------------------------------
# Normalización de nombres de columnas (robusta)
# -------------------------------------------------
def _normalize_colname(c: str) -> str:
    return str(c).strip().lower().replace(" ", "_")

issuer_map_raw.columns = [_normalize_colname(c) for c in issuer_map_raw.columns]

# intentamos mapear columnas típicas a: issuer, ticker, sector
rename_map = {}

# issuer
issuer_candidates = ["issuer", "issuer_name", "issuername", "company", "company_name", "name"]
for c in issuer_candidates:
    if c in issuer_map_raw.columns:
        rename_map[c] = "issuer"
        break

# ticker
ticker_candidates = ["ticker", "ric", "equity_ric", "equityric"]
for c in ticker_candidates:
    if c in issuer_map_raw.columns:
        rename_map[c] = "ticker"
        break

# sector
sector_candidates = ["sector", "gics_sector", "gics", "trbc_sector", "industry"]
for c in sector_candidates:
    if c in issuer_map_raw.columns:
        rename_map[c] = "sector"
        break

issuer_map = issuer_map_raw.rename(columns=rename_map).copy()

if "issuer" not in issuer_map.columns:
    raise KeyError(
        "No pude detectar columna issuer en los excels. "
        "Asegurate de tener una columna tipo 'issuer' o 'company_name'."
    )

# Mantener solo columnas relevantes si existen
keep = ["issuer"]
if "ticker" in issuer_map.columns:
    keep.append("ticker")
if "sector" in issuer_map.columns:
    keep.append("sector")

issuer_map = issuer_map[keep].copy()

# Limpieza mínima (sin inventar información)
issuer_map["issuer"] = issuer_map["issuer"].astype(str).str.strip()

if "ticker" in issuer_map.columns:
    issuer_map["ticker"] = issuer_map["ticker"].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA})
if "sector" in issuer_map.columns:
    issuer_map["sector"] = issuer_map["sector"].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA})

# Drop issuer vacíos + dedup por issuer (conservador)
issuer_map = issuer_map.replace({"": pd.NA}).dropna(subset=["issuer"])
issuer_map = issuer_map.drop_duplicates(subset=["issuer"], keep="first").reset_index(drop=True)

# -------------------------------------------------
# QA rápido
# -------------------------------------------------
print("\n=== issuer_map (preview) ===")
print(issuer_map.head(10))
print("\n=== issuer_map (QA) ===")
print("n_issuer:", issuer_map["issuer"].nunique())
if "ticker" in issuer_map.columns:
    print("ticker NA%:", issuer_map["ticker"].isna().mean())
if "sector" in issuer_map.columns:
    print("sector NA%:", issuer_map["sector"].isna().mean())

# -------------------------------------------------
# Export CLEAN
# -------------------------------------------------
OUT_MAP_PARQ = CLEAN / "issuer_ticker_sector.parquet"
OUT_MAP_CSV  = CLEAN / "issuer_ticker_sector.csv"

issuer_map.to_parquet(OUT_MAP_PARQ, index=False)
issuer_map.to_csv(OUT_MAP_CSV, index=False)

print(f"\n✅ Mapping guardado:")
print(f"- {OUT_MAP_PARQ}")
print(f"- {OUT_MAP_CSV}")

## 7. Merge spreads + mapping

Input:
- `bonds_monthly_spreads_firmlevel.parquet`
- `issuer_ticker_sector.parquet`

Output:
- `data/intermediate/spreads_firm_enriched.parquet`

Objetivo:
- Incorporar ticker y sector al panel firm-month de spreads.
- No modificar definiciones de spreads.
- No alterar número de observaciones (salvo por joins left).

Chequeos:
- No se pierden filas del spread firm-level.
- % de ticker NA razonable.
- Llave: issuer-date.

In [ ]:
# ==========================================================
# 7. Merge spreads firm-level + mapping issuer→ticker/sector
# ==========================================================

import pandas as pd

# -------------------------------------------------
# Cargar spreads firm-level
# -------------------------------------------------
SPREADS_FIRM_PARQ = INTERIM / "bonds_monthly_spreads_firmlevel.parquet"
SPREADS_FIRM_CSV  = INTERIM / "bonds_monthly_spreads_firmlevel.csv"

if SPREADS_FIRM_PARQ.exists():
    spreads_firm = pd.read_parquet(SPREADS_FIRM_PARQ)
elif SPREADS_FIRM_CSV.exists():
    spreads_firm = pd.read_csv(SPREADS_FIRM_CSV)
else:
    raise FileNotFoundError("No encontré bonds_monthly_spreads_firmlevel en INTERIM.")

spreads_firm["date"] = pd.to_datetime(spreads_firm["date"], errors="coerce")

print("Spreads firm-level shape:", spreads_firm.shape)

# -------------------------------------------------
# Cargar mapping issuer→ticker→sector
# -------------------------------------------------
MAP_PARQ = CLEAN / "issuer_ticker_sector.parquet"
MAP_CSV  = CLEAN / "issuer_ticker_sector.csv"

if MAP_PARQ.exists():
    issuer_map = pd.read_parquet(MAP_PARQ)
elif MAP_CSV.exists():
    issuer_map = pd.read_csv(MAP_CSV)
else:
    raise FileNotFoundError("No encontré issuer_ticker_sector en CLEAN.")

print("Mapping shape:", issuer_map.shape)

# -------------------------------------------------
# Merge LEFT (no perder observaciones de spreads)
# -------------------------------------------------
n_before = spreads_firm.shape[0]

spreads_firm_enriched = spreads_firm.merge(
    issuer_map,
    on="issuer",
    how="left"
)

n_after = spreads_firm_enriched.shape[0]

# -------------------------------------------------
# Chequeos
# -------------------------------------------------
print("\n=== QA Merge ===")
print("Filas antes:", n_before)
print("Filas después:", n_after)
print("Δ filas:", n_after - n_before)

assert n_before == n_after, "El merge cambió el número de filas (revisar claves)."

# % NA ticker / sector
if "ticker" in spreads_firm_enriched.columns:
    print("Ticker NA%:", spreads_firm_enriched["ticker"].isna().mean())

if "sector" in spreads_firm_enriched.columns:
    print("Sector NA%:", spreads_firm_enriched["sector"].isna().mean())

# Chequeo duplicados issuer-date
dup = spreads_firm_enriched.duplicated(subset=["issuer", "date"]).sum()
print("Duplicados issuer-date:", dup)
assert dup == 0, "Hay duplicados issuer-date tras el merge."

print("\nPreview:")
print(spreads_firm_enriched.head())

# -------------------------------------------------
# Guardar intermedio
# -------------------------------------------------
OUT_ENRICHED_PARQ = INTERIM / "spreads_firm_enriched.parquet"
OUT_ENRICHED_CSV  = INTERIM / "spreads_firm_enriched.csv"

spreads_firm_enriched.to_parquet(OUT_ENRICHED_PARQ, index=False)
spreads_firm_enriched.to_csv(OUT_ENRICHED_CSV, index=False)

print("\n✅ spreads_firm_enriched guardado en:")
print("-", OUT_ENRICHED_PARQ)
print("-", OUT_ENRICHED_CSV)

## 8. Beta e IVOL (Rolling CAPM)

Input:
- `equity_market_returns.parquet`
- `market_sp500.parquet` (retorno mercado)

Output:
- `data/intermediate/equity_features_monthly.parquet`

Metodología:
- Retornos diarios.
- Rolling CAPM por firma.
- Ventana exactamente igual a notebook original.
- Beta = coeficiente sobre retorno de mercado.
- IVOL = desviación estándar del residuo CAPM.

Chequeos:
- No modificar ventana.
- No modificar frecuencia.
- No modificar definición de retorno.

In [ ]:
# ==========================================================
# 8. Rolling CAPM → Beta e IVOL
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

# -------------------------------------------------
# Load equity daily returns
# -------------------------------------------------
EQ_PARQ = RAW / "equity_market_returns.parquet"
EQ_CSV  = RAW / "equity_market_returns.csv"

if EQ_PARQ.exists():
    equity = pd.read_parquet(EQ_PARQ)
elif EQ_CSV.exists():
    equity = pd.read_csv(EQ_CSV)
else:
    raise FileNotFoundError("No encontré equity_market_returns en RAW.")

equity["date"] = pd.to_datetime(equity["date"], errors="coerce")
equity = equity.sort_values(["ticker", "date"]).copy()

# -------------------------------------------------
# Load market returns
# -------------------------------------------------
MKT_PARQ = RAW / "market_sp500.parquet"
MKT_CSV  = RAW / "market_sp500.csv"

if MKT_PARQ.exists():
    market = pd.read_parquet(MKT_PARQ)
elif MKT_CSV.exists():
    market = pd.read_csv(MKT_CSV)
else:
    raise FileNotFoundError("No encontré market_sp500 en RAW.")

market["date"] = pd.to_datetime(market["date"], errors="coerce")

# -------------------------------------------------
# Merge equity + market
# -------------------------------------------------
df = equity.merge(
    market[["date", "mkt_ret"]],
    on="date",
    how="left"
)

df = df.sort_values(["ticker", "date"]).copy()

# -------------------------------------------------
# Parámetro rolling (MANTENER EXACTAMENTE TU VENTANA)
# -------------------------------------------------
ROLL_WINDOW = 252  # ⚠️ cambiar solo si tu notebook usa otro valor

results = []

# -------------------------------------------------
# Rolling CAPM por ticker
# -------------------------------------------------
for ticker, sub in df.groupby("ticker"):

    sub = sub.dropna(subset=["ret", "mkt_ret"]).copy()
    sub = sub.sort_values("date")

    if len(sub) < ROLL_WINDOW:
        continue

    for i in range(ROLL_WINDOW, len(sub)):

        window = sub.iloc[i-ROLL_WINDOW:i]

        y = window["ret"]
        X = add_constant(window["mkt_ret"])

        try:
            model = OLS(y, X).fit()
        except:
            continue

        beta = model.params["mkt_ret"]
        resid_std = model.resid.std()

        results.append({
            "ticker": ticker,
            "date": sub.iloc[i]["date"],
            "beta": beta,
            "ivol": resid_std
        })

equity_features = pd.DataFrame(results)

# -------------------------------------------------
# Pasar a frecuencia mensual (igual que notebook)
# -------------------------------------------------
equity_features["date"] = pd.to_datetime(equity_features["date"])
equity_features["month"] = equity_features["date"].dt.to_period("M")

equity_features_monthly = (
    equity_features
    .groupby(["ticker", "month"], as_index=False)
    .agg({
        "beta": "mean",
        "ivol": "mean"
    })
)

equity_features_monthly["date"] = equity_features_monthly["month"].dt.to_timestamp()
equity_features_monthly = equity_features_monthly.drop(columns=["month"])

# -------------------------------------------------
# QA
# -------------------------------------------------
print("Equity features shape:", equity_features_monthly.shape)
print("N tickers:", equity_features_monthly["ticker"].nunique())
print("Date range:", equity_features_monthly["date"].min(), "→", equity_features_monthly["date"].max())

# -------------------------------------------------
# Export INTERMEDIATE
# -------------------------------------------------
OUT_EQ_PARQ = INTERIM / "equity_features_monthly.parquet"
OUT_EQ_CSV  = INTERIM / "equity_features_monthly.csv"

equity_features_monthly.to_parquet(OUT_EQ_PARQ, index=False)
equity_features_monthly.to_csv(OUT_EQ_CSV, index=False)

print("✅ equity_features_monthly guardado en:")
print("-", OUT_EQ_PARQ)
print("-", OUT_EQ_CSV)

## 9. Alternativas de volatilidad

### 9.1 IVOL sectorial (ETF sectorial)

Input:
- `data/raw/sector_etfs_daily.csv` con columnas: `date`, `price`, `sector_ric`

Output:
- `data/intermediate/ivol_sector_monthly_60d.parquet`

Definición:
- ret diario = log(price).diff por `sector_ric`
- ivol_sector_daily = rolling std(ret) por `sector_ric` (WINDOW=60, MIN_OBS=45)
- mensualización = promedio del ivol daily dentro del mes (fin de mes)

Chequeos:
- columnas requeridas
- cobertura temporal y NA

---

### 9.2 Volatilidad del mercado (S&P 500)

Input:
- `data/raw/market_sp500.parquet` (o .csv)

Output:
- `data/intermediate/mkt_vol_monthly_60d.parquet`

Definición:
- si existe retorno: usa `mkt_ret`
- si no, construye `mkt_ret` = log(precio).diff
- mkt_vol_daily = rolling std(mkt_ret) (WINDOW=60, MIN_OBS=45)
- mensualización = promedio del mkt_vol daily dentro del mes (fin de mes)

Chequeos:
- columnas reconocibles (ret o precio)
- rango temporal + plausibilidad básica

In [ ]:
# ==========================================================
# 9.1 Alternativa: Volatilidad Sectorial (ETF sectorial)
# Input: sector_etfs_daily.csv con columnas:
#   - date
#   - price
#   - sector_ric (RIC del ETF sectorial)
# Output: ivol_sector_monthly_{WINDOW}d.parquet
# ==========================================================

import pandas as pd
import numpy as np

WINDOW = 60
MIN_OBS = 45

# -----------------------------
# 1) Load
# -----------------------------
path = RAW / "sector_etfs_daily.csv"
if not path.exists():
    raise FileNotFoundError(f"No encontré {path}")

etf = pd.read_csv(path)

required = {"date", "price", "sector_ric"}
if not required.issubset(etf.columns):
    raise ValueError(f"Faltan columnas. Requeridas: {required}. Tiene: {list(etf.columns)}")

# parse
etf["date"] = pd.to_datetime(etf["date"], errors="coerce")
etf["price"] = pd.to_numeric(etf["price"], errors="coerce")
etf["sector_ric"] = etf["sector_ric"].astype(str)

# limpieza
etf = etf.dropna(subset=["date", "price", "sector_ric"]).copy()
etf = etf.sort_values(["sector_ric", "date"])

# -----------------------------
# 2) Retornos diarios por sector (log-ret)
# -----------------------------
etf["ret"] = (
    etf.groupby("sector_ric")["price"]
       .apply(lambda s: np.log(s).diff())
       .reset_index(level=0, drop=True)
)

# -----------------------------
# 3) IVOL sectorial daily = rolling std(ret)
# -----------------------------
etf["ivol_sector_daily"] = (
    etf.groupby("sector_ric")["ret"]
       .rolling(WINDOW, min_periods=MIN_OBS)
       .std()
       .reset_index(level=0, drop=True)
)

# -----------------------------
# 4) Mensualizar (fin de mes)
# -----------------------------
etf["month"] = etf["date"].dt.to_period("M").dt.to_timestamp("M")

ivol_sector_monthly = (
    etf.groupby(["sector_ric", "month"], as_index=False)
       .agg(ivol_sector=("ivol_sector_daily", "mean"))
       .rename(columns={"month": "date"})
)

# -----------------------------
# 5) Guardar
# -----------------------------
out_path = INTERIM / f"ivol_sector_monthly_{WINDOW}d.parquet"
ivol_sector_monthly.to_parquet(out_path, index=False)

print("✅ IVOL sectorial mensual guardada en:", out_path)
print(ivol_sector_monthly.head())

# -----------------------------
# Diagnóstico (igual estilo que tu notebook)
# -----------------------------
df = ivol_sector_monthly.copy()

print("\n--- Cobertura ---")
print("Sectores (sector_ric):", df["sector_ric"].nunique())
print("Obs:", len(df))
print("Rango:", df["date"].min(), "->", df["date"].max())

print("\n--- Missingness ---")
print("NaN ivol_sector:", df["ivol_sector"].isna().sum())

print("\n--- Estadísticos ---")
print(df["ivol_sector"].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

print("\n--- Checks de plausibilidad ---")
print("IVOL <= 0:", (df["ivol_sector"] <= 0).sum())
p95 = df["ivol_sector"].quantile(0.95)
print("IVOL p95:", round(float(p95), 6))

print("\n--- Sectores más volátiles (promedio) ---")
print(df.groupby("sector_ric")["ivol_sector"].mean().sort_values(ascending=False).head(10))

print("\n--- Sectores menos volátiles (promedio) ---")
print(df.groupby("sector_ric")["ivol_sector"].mean().sort_values().head(10))

print("\n--- Ejemplo: meses con IVOL extrema ---")
print(df.sort_values("ivol_sector", ascending=False).head(10)[["sector_ric", "date", "ivol_sector"]])

In [ ]:
# ==========================================================
# 9.2 Market volatility (S&P 500): rolling std de retornos diarios
# Output: mkt_vol_monthly_{WINDOW}d.parquet con columna mkt_vol_{WINDOW}d
# ==========================================================

import pandas as pd
import numpy as np

WINDOW = 60
MIN_OBS = 45

# -----------------------------
# 1) Load market_sp500
# -----------------------------
mkt_parquet = RAW / "market_sp500.parquet"
mkt_csv     = RAW / "market_sp500.csv"

if mkt_parquet.exists():
    mkt = pd.read_parquet(mkt_parquet)
elif mkt_csv.exists():
    mkt = pd.read_csv(mkt_csv)
else:
    raise FileNotFoundError("No encontré market_sp500 en RAW (ni parquet ni csv).")

# -----------------------------
# 2) Normalizar columnas
# -----------------------------
if "date" not in mkt.columns:
    raise ValueError(f"market_sp500 debe tener columna 'date'. Tiene: {list(mkt.columns)}")

mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")
mkt = mkt.sort_values("date").copy()

# detectar si trae retornos o precio
ret_candidates = ["mkt_ret", "ret", "return", "sp500_ret", "daily_ret"]
price_candidates = ["price", "close", "px_last", "index", "spx", "value"]

ret_col = next((c for c in ret_candidates if c in mkt.columns), None)
price_col = next((c for c in price_candidates if c in mkt.columns), None)

if ret_col is None and price_col is None:
    raise ValueError(
        "market_sp500 no tiene retornos ni precio reconocibles.\n"
        f"Columnas: {list(mkt.columns)}\n"
        f"Busqué returns en: {ret_candidates}\n"
        f"Busqué price en: {price_candidates}"
    )

# -----------------------------
# 3) Construir retorno diario
# -----------------------------
mkt = mkt.copy()

if ret_col is not None:
    mkt["mkt_ret"] = pd.to_numeric(mkt[ret_col], errors="coerce")
else:
    mkt["price"] = pd.to_numeric(mkt[price_col], errors="coerce")
    mkt["mkt_ret"] = np.log(mkt["price"]).diff()   # log-ret

mkt = mkt.replace([np.inf, -np.inf], np.nan)
mkt = mkt.dropna(subset=["mkt_ret"]).copy()

# -----------------------------
# 4) Volatilidad rolling diaria
# -----------------------------
mkt["mkt_vol_daily"] = (
    mkt["mkt_ret"]
    .rolling(WINDOW, min_periods=MIN_OBS)
    .std()
)

# -----------------------------
# 5) Mensualizar (fin de mes)
# -----------------------------
mkt["month"] = mkt["date"].dt.to_period("M").dt.to_timestamp("M")

mkt_vol_monthly = (
    mkt.groupby("month", as_index=False)
       .agg(mkt_vol=("mkt_vol_daily", "mean"))
       .rename(columns={"month": "date"})
)

mkt_vol_monthly = mkt_vol_monthly.rename(columns={"mkt_vol": f"mkt_vol_{WINDOW}d"})

# -----------------------------
# 6) Guardar
# -----------------------------
out_path = INTERIM / f"mkt_vol_monthly_{WINDOW}d.parquet"
mkt_vol_monthly.to_parquet(out_path, index=False)

col = f"mkt_vol_{WINDOW}d"
df = mkt_vol_monthly.copy()

print("✅ mkt_vol mensual guardada en:", out_path)
print(df.head(10))
print("\nRango:", df["date"].min(), "->", df["date"].max())
print("\nStats:")
print(df[col].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

print("\nNaNs:", df[col].isna().sum())
print("p95:", df[col].quantile(0.95))
print("max:", df[col].max())

print("\nTop 10 meses más volátiles:")
print(df.sort_values(col, ascending=False).head(10))

## 10. Fundamentals trimestrales → mensuales

Input:
- `fundamentals_extended_q.parquet`

Output:
- `data/intermediate/fundamentals_monthly_enriched.parquet`

Metodología:
- Datos trimestrales.
- Forward-fill a frecuencia mensual.
- Mantener exactamente las transformaciones originales.
- No modificar definiciones de leverage, size, liquidity, etc.

Chequeos:
- No perder firmas.
- Rango temporal consistente.
- Sin duplicados ticker-date.

In [ ]:
# ==========================================================
# 10. Fundamentals Q → M
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------------------------
# Load fundamentals trimestrales
# -------------------------------------------------
FUND_PARQ = RAW / "fundamentals_extended_q.parquet"
FUND_CSV  = RAW / "fundamentals_extended_q.csv"

if FUND_PARQ.exists():
    fundamentals_q = pd.read_parquet(FUND_PARQ)
elif FUND_CSV.exists():
    fundamentals_q = pd.read_csv(FUND_CSV)
else:
    raise FileNotFoundError("No encontré fundamentals_extended_q en RAW.")

fundamentals_q["date"] = pd.to_datetime(fundamentals_q["date"], errors="coerce")

fundamentals_q = fundamentals_q.sort_values(["ticker", "date"]).copy()

print("Fundamentals trimestrales:", fundamentals_q.shape)

# -------------------------------------------------
# Forward fill trimestral a mensual
# -------------------------------------------------
fundamentals_q["month"] = fundamentals_q["date"].dt.to_period("M")

fund_monthly = (
    fundamentals_q
    .set_index("date")
    .groupby("ticker")
    .resample("M")
    .ffill()
    .reset_index()
)

# -------------------------------------------------
# Mantener mismas transformaciones que tu notebook
# (NO cambiar definiciones)
# -------------------------------------------------

# Ejemplos típicos que ya tenías:
if "total_debt" in fund_monthly.columns and "total_assets" in fund_monthly.columns:
    fund_monthly["leverage"] = fund_monthly["total_debt"] / fund_monthly["total_assets"]

if "total_assets" in fund_monthly.columns:
    fund_monthly["log_assets"] = np.log(fund_monthly["total_assets"])

if "cash" in fund_monthly.columns and "total_assets" in fund_monthly.columns:
    fund_monthly["cash_to_assets"] = fund_monthly["cash"] / fund_monthly["total_assets"]

if "current_assets" in fund_monthly.columns and "current_liabilities" in fund_monthly.columns:
    fund_monthly["current_ratio"] = fund_monthly["current_assets"] / fund_monthly["current_liabilities"]

if "net_income" in fund_monthly.columns and "total_assets" in fund_monthly.columns:
    fund_monthly["roa"] = fund_monthly["net_income"] / fund_monthly["total_assets"]

# -------------------------------------------------
# QA
# -------------------------------------------------
print("Fundamentals mensuales:", fund_monthly.shape)
print("N firms:", fund_monthly["ticker"].nunique())
print("Date range:", fund_monthly["date"].min(), "→", fund_monthly["date"].max())

dup = fund_monthly.duplicated(subset=["ticker", "date"]).sum()
assert dup == 0, "Hay duplicados ticker-date en fundamentals."

# -------------------------------------------------
# Export INTERMEDIATE
# -------------------------------------------------
OUT_FUND_PARQ = INTERIM / "fundamentals_monthly_enriched.parquet"
OUT_FUND_CSV  = INTERIM / "fundamentals_monthly_enriched.csv"

fund_monthly.to_parquet(OUT_FUND_PARQ, index=False)
fund_monthly.to_csv(OUT_FUND_CSV, index=False)

print("✅ fundamentals_monthly_enriched guardado en:")
print("-", OUT_FUND_PARQ)
print("-", OUT_FUND_CSV)

## 11. Poder de mercado (Market Share)

Input:
- `data/intermediate/fundamentals_monthly_enriched.parquet`
- (opcional) `data/clean/issuer_ticker_sector.parquet` si el sector no está en fundamentals

Output:
- `data/intermediate/market_power_monthly.parquet`

Definición:
- Market share = ventas (firma) / ventas totales (sector × mes)
- Se construye a frecuencia mensual (firm-month)

Chequeos:
- No duplicados ticker-date
- Market share en [0,1] (salvo casos por missing/sector mal asignado)
- Cobertura temporal consistente con el panel

In [ ]:
# ==========================================================
# 11. Poder de mercado (Market Share) – mensual
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------------------------
# 1) Load fundamentals mensuales enriquecidos
# -------------------------------------------------
FUND_M_PARQ = INTERIM / "fundamentals_monthly_enriched.parquet"
FUND_M_CSV  = INTERIM / "fundamentals_monthly_enriched.csv"

if FUND_M_PARQ.exists():
    fund = pd.read_parquet(FUND_M_PARQ)
elif FUND_M_CSV.exists():
    fund = pd.read_csv(FUND_M_CSV)
else:
    raise FileNotFoundError("No encontré fundamentals_monthly_enriched en INTERIM.")

fund["date"] = pd.to_datetime(fund["date"], errors="coerce")

# -------------------------------------------------
# 2) Asegurar sector (si no está, mergear mapping)
# -------------------------------------------------
sector_candidates = ["sector", "gics_sector", "trbc_sector", "industry", "sector_name"]
sector_col = next((c for c in sector_candidates if c in fund.columns), None)

if sector_col is None:
    # Merge con mapping issuer_ticker_sector (si el sector vive ahí)
    MAP_PARQ = CLEAN / "issuer_ticker_sector.parquet"
    MAP_CSV  = CLEAN / "issuer_ticker_sector.csv"

    if MAP_PARQ.exists():
        issuer_map = pd.read_parquet(MAP_PARQ)
    elif MAP_CSV.exists():
        issuer_map = pd.read_csv(MAP_CSV)
    else:
        raise FileNotFoundError("No encontré issuer_ticker_sector en CLEAN para traer sector.")

    if "ticker" not in issuer_map.columns:
        raise KeyError("issuer_ticker_sector no tiene columna ticker.")

    # Intentar detectar sector en el mapping
    sector_col_map = next((c for c in sector_candidates if c in issuer_map.columns), None)
    if sector_col_map is None:
        raise KeyError(
            "No encontré columna sector en fundamentals ni en issuer_ticker_sector.\n"
            f"Busqué: {sector_candidates}"
        )

    fund = fund.merge(
        issuer_map[["ticker", sector_col_map]].drop_duplicates("ticker"),
        on="ticker",
        how="left"
    )
    sector_col = sector_col_map  # ahora viene del mapping

# Normalizar nombre a "sector" para consistencia interna
if sector_col != "sector":
    fund = fund.rename(columns={sector_col: "sector"})
else:
    fund = fund.rename(columns={"sector": "sector"})

# -------------------------------------------------
# 3) Detectar columna de ventas (revenue/sales)
# -------------------------------------------------
sales_candidates = ["revenue", "sales", "net_sales", "total_revenue", "rev", "sales_revenue"]
sales_col = next((c for c in sales_candidates if c in fund.columns), None)

if sales_col is None:
    raise KeyError(
        "No encontré columna de ventas en fundamentals.\n"
        f"Busqué: {sales_candidates}\n"
        f"Columnas disponibles: {list(fund.columns)[:60]} ..."
    )

fund[sales_col] = pd.to_numeric(fund[sales_col], errors="coerce")

# -------------------------------------------------
# 4) Construir market share por sector-mes
# -------------------------------------------------
tmp = fund[["ticker", "date", "sector", sales_col]].copy()
tmp = tmp.dropna(subset=["ticker", "date", "sector"]).copy()

# ventas sectoriales por mes
sector_totals = (
    tmp.groupby(["sector", "date"], as_index=False)[sales_col]
       .sum()
       .rename(columns={sales_col: "sector_sales_total"})
)

mp = tmp.merge(sector_totals, on=["sector", "date"], how="left")

mp["market_share"] = mp[sales_col] / mp["sector_sales_total"]

# -------------------------------------------------
# 5) QA básico
# -------------------------------------------------
dup = mp.duplicated(subset=["ticker", "date"]).sum()
assert dup == 0, f"Duplicados ticker-date en market power: {dup}"

print("✅ market power shape:", mp.shape)
print("✅ n firms:", mp["ticker"].nunique())
print("✅ rango:", mp["date"].min(), "→", mp["date"].max())
print("Market share NA%:", mp["market_share"].isna().mean())

# plausibilidad
neg = (mp["market_share"] < 0).sum()
gt1 = (mp["market_share"] > 1).sum()
print("market_share < 0:", neg)
print("market_share > 1:", gt1)

print("\nTop 10 market_share (para ver outliers):")
print(mp.sort_values("market_share", ascending=False).head(10)[["ticker", "date", "sector", "market_share"]])

# -------------------------------------------------
# 6) Export intermedio
# -------------------------------------------------
OUT_MP_PARQ = INTERIM / "market_power_monthly.parquet"
OUT_MP_CSV  = INTERIM / "market_power_monthly.csv"

mp_out = mp[["ticker", "date", "sector", "market_share"]].copy()
mp_out.to_parquet(OUT_MP_PARQ, index=False)
mp_out.to_csv(OUT_MP_CSV, index=False)

print("\n✅ market_power_monthly guardado en:")
print("-", OUT_MP_PARQ)
print("-", OUT_MP_CSV)

## 12. Factor de crédito agregado (Alternativa CRC – iBoxx)

Input:
- `data/intermediate/bonds_monthly_spreads_firmlevel.parquet`
- `data/raw/iboxx_credit_indices_monthly.parquet`

Output:
- `data/intermediate/crc_alt_iboxx_betas_monthly.parquet`

Definición:
- Construimos shocks agregados de crédito desde índices iBoxx (log-returns):
  - `credit_level` = -Δlog(iboxx_ig_liquid)
  - `credit_slope` = [-Δlog(iboxx_overall_10y_plus)] − [-Δlog(iboxx_overall_1_5y)]
- Estimamos exposición firm-level (rolling OLS):
  - y = Δ spread_mean_bps por issuer (`d_spread`)
  - X = `credit_level`, `credit_slope` (+ constante)
  - ventana = 36 meses, mínimo = 24 obs
- Guardamos betas: `crc_level_beta`, `crc_slope_beta` + métricas (R², nobs)

Chequeos:
- Series iBoxx requeridas presentes
- Cobertura temporal razonable
- Distribución de betas y R²

In [ ]:
# ==========================================================
# 12. CRC alternativa (iBoxx): betas firm-level a shocks agregados
# Lee:  iboxx_credit_indices_monthly.parquet
# Output: crc_alt_iboxx_betas_monthly.parquet
# ==========================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm

# -----------------------------
# 1) Cargar spreads firm-level
# -----------------------------
SP_FIRM = INTERIM / "bonds_monthly_spreads_firmlevel.parquet"
if not SP_FIRM.exists():
    raise FileNotFoundError(f"No encontré {SP_FIRM}")

sp = pd.read_parquet(SP_FIRM)
sp["date"] = pd.to_datetime(sp["date"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")
sp = sp.sort_values(["issuer", "date"]).copy()

if "spread_mean_bps" not in sp.columns:
    raise KeyError("bonds_monthly_spreads_firmlevel.parquet debe tener 'spread_mean_bps'.")

# Δ spread por firma (target para CRC)
sp["d_spread"] = sp.groupby("issuer")["spread_mean_bps"].diff()

# -----------------------------
# 2) Cargar iBoxx monthly
# -----------------------------
IBOXX_PATH = RAW / "iboxx_credit_indices_monthly.parquet"
if not IBOXX_PATH.exists():
    raise FileNotFoundError(f"No encontré {IBOXX_PATH}")

iboxx = pd.read_parquet(IBOXX_PATH)
iboxx["date"] = pd.to_datetime(iboxx["date"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")

# Normalizar (por si viniera como close)
if "value" not in iboxx.columns and "close" in iboxx.columns:
    iboxx = iboxx.rename(columns={"close": "value"})

# Asegurar que tenemos lo necesario
req_cols = {"date", "ric", "series", "value"}
miss = req_cols - set(iboxx.columns)
if miss:
    raise ValueError(
        f"Faltan columnas en iboxx_credit_indices_monthly.parquet: {miss}. "
        f"Tengo: {iboxx.columns.tolist()}"
    )

# Pasar a wide por series (nombres amigables ya creados en tu descarga)
wide = (
    iboxx.pivot(index="date", columns="series", values="value")
         .sort_index()
)

need = ["iboxx_ig_liquid", "iboxx_overall_1_5y", "iboxx_overall_10y_plus"]
missing = [c for c in need if c not in wide.columns]
if missing:
    raise ValueError(f"Faltan series requeridas en iBoxx: {missing}. Disponibles: {list(wide.columns)}")

# -----------------------------
# 3) Construir shocks agregados (log-returns)
# -----------------------------
ret = np.log(wide[need]).diff()

# Shock de nivel: empeora crédito si cae el índice => shock positivo
credit_level = -ret["iboxx_ig_liquid"]

# Shock de pendiente: largo - corto (empeora más en largo que en corto)
credit_slope = (-ret["iboxx_overall_10y_plus"]) - (-ret["iboxx_overall_1_5y"])

factors = pd.DataFrame({
    "date": credit_level.index,
    "credit_level": credit_level.values,
    "credit_slope": credit_slope.values
}).dropna()

# -----------------------------
# 4) Rolling betas por issuer
# -----------------------------
WINDOW  = 36   # meses
MIN_OBS = 24

rows = []

for issuer, g in sp.groupby("issuer"):
    tmp = (
        g[["date", "d_spread"]]
        .merge(factors, on="date", how="inner")
        .dropna(subset=["d_spread", "credit_level", "credit_slope"])
        .sort_values("date")
    )

    if len(tmp) < MIN_OBS:
        continue

    for i in range(MIN_OBS, len(tmp) + 1):
        sub = tmp.iloc[max(0, i - WINDOW):i]
        if len(sub) < MIN_OBS:
            continue

        X = sm.add_constant(sub[["credit_level", "credit_slope"]])
        y = sub["d_spread"]

        try:
            res = sm.OLS(y, X).fit()
        except Exception:
            continue

        rows.append({
            "issuer": issuer,
            "date": sub["date"].iloc[-1],
            "crc_level_beta": float(res.params.get("credit_level", np.nan)),
            "crc_slope_beta": float(res.params.get("credit_slope", np.nan)),
            "crc_r2": float(res.rsquared),
            "crc_nobs": int(res.nobs),
        })

crc_alt = pd.DataFrame(rows).sort_values(["issuer", "date"])

# -----------------------------
# 5) Guardar
# -----------------------------
out_path = INTERIM / "crc_alt_iboxx_betas_monthly.parquet"
crc_alt.to_parquet(out_path, index=False)

print("✅ CRC alternativa (iBoxx) guardada en:", out_path)
print("Issuers:", crc_alt["issuer"].nunique(), "| filas:", len(crc_alt))
print("Rango:", crc_alt["date"].min(), "->", crc_alt["date"].max())
print(crc_alt[["crc_level_beta", "crc_slope_beta", "crc_r2", "crc_nobs"]].describe())

# -----------------------------
# 6) Diagnóstico (igual estilo que tu notebook)
# -----------------------------
print("\n" + "="*78)
print("SECCIÓN 6 — DIAGNÓSTICO CRC ALTERNATIVA (iBoxx)")
print("="*78)

print("\n[6.1] Inputs: spreads firm-level e iBoxx")
print("-"*78)
print("Spreads firm-level (sp):")
print("  filas:", len(sp), "| issuers:", sp["issuer"].nunique())
print("  rango:", sp["date"].min(), "->", sp["date"].max())
print("  spread_mean_bps NaN%:", sp["spread_mean_bps"].isna().mean())
print("  d_spread NaN% (esperable por diff):", sp["d_spread"].isna().mean())

print("\niBoxx (wide):")
print("  need:", need)
print("  rango:", wide.index.min(), "->", wide.index.max())
print("  NaN% por serie (level):")
print((wide[need].isna().mean() * 100).round(2))

print("\n[6.2] Factores (credit_level, credit_slope)")
print("-"*78)
print("Factores: filas:", len(factors), "| rango:", factors["date"].min(), "->", factors["date"].max())
print("\nStats credit_level / credit_slope:")
print(
    factors[["credit_level", "credit_slope"]]
      .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)
print("\nCorrelación entre factores:")
print(factors[["credit_level", "credit_slope"]].corr())

print("\n[6.3] Cobertura CRC estimada")
print("-"*78)
print("  filas:", len(crc_alt), "| issuers:", crc_alt["issuer"].nunique())
print("  rango:", crc_alt["date"].min(), "->", crc_alt["date"].max())

obs_per_issuer = crc_alt.groupby("issuer").size()
print("\nObs por issuer (resumen):")
print(obs_per_issuer.describe()[["min", "25%", "50%", "mean", "75%", "max"]])

print("\nDistribución crc_nobs:")
print(crc_alt["crc_nobs"].value_counts().sort_index())

print("\n[6.4] Calidad (R²)")
print("-"*78)
print(crc_alt["crc_r2"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\n[6.5] Outliers (P1/P99)")
print("-"*78)
def outlier_block(df, col):
    p1, p99 = df[col].quantile([0.01, 0.99])
    share = ((df[col] < p1) | (df[col] > p99)).mean()
    print(f"{col}: P1={p1:,.4f} | P99={p99:,.4f} | % fuera={share*100:.2f}%")

outlier_block(crc_alt, "crc_level_beta")
outlier_block(crc_alt, "crc_slope_beta")

print("\n[6.6] Estabilidad temporal (promedios por año)")
print("-"*78)
tmp_t = crc_alt.copy()
tmp_t["year"] = pd.to_datetime(tmp_t["date"]).dt.year
year_means = tmp_t.groupby("year")[["crc_level_beta", "crc_slope_beta", "crc_r2"]].mean()
year_counts = tmp_t.groupby("year").size().rename("n_obs")
print(year_means.merge(year_counts, left_index=True, right_index=True))

print("\n[6.7] Chequeo de signo (orientativo)")
print("-"*78)
sign_share = (crc_alt["crc_level_beta"] > 0).mean()
print(f"Share crc_level_beta > 0: {sign_share*100:.2f}%")

print("\n✅ Diagnóstico CRC alternativa finalizado.")
print("="*78)

## 13. Merge master (panel firm-month)

Input:
- `data/intermediate/spreads_firm_enriched.parquet`  (backbone)
- `data/intermediate/equity_features_monthly.parquet`
- `data/intermediate/fundamentals_monthly_enriched.parquet`
- `data/intermediate/market_power_monthly.parquet`
- `data/intermediate/crc_alt_iboxx_betas_monthly.parquet`

Output:
- `data/intermediate/panel_master_pre_filters.parquet`

Objetivo:
- Construir el panel firm-month con todas las variables explicativas.
- Backbone = spreads firm-month (no perder filas).
- Merges left para preservar muestra.
- Normalización de fecha: fin de mes (EOM) para todas las fuentes.

Chequeos:
- Llave única `issuer-date` en el panel final.
- Conteo de filas no cambia respecto al backbone.
- Reporte NA% por bloque (equity/fundamentals/market_power/credit).

In [ ]:
# ==========================================================
# 13. Merge master (panel firm-month)
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def to_eom(s: pd.Series) -> pd.Series:
    """Convierte fechas a fin de mes (timestamp M)."""
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")

def load_df(parq: Path, csv: Path | None = None) -> pd.DataFrame:
    if parq.exists():
        return pd.read_parquet(parq)
    if csv is not None and csv.exists():
        return pd.read_csv(csv)
    raise FileNotFoundError(f"No encontré {parq} (ni csv fallback).")

def na_report(df: pd.DataFrame, cols: list[str], label: str) -> None:
    cols = [c for c in cols if c in df.columns]
    if not cols:
        print(f"[{label}] (sin columnas reportables)")
        return
    rep = (df[cols].isna().mean() * 100).round(2).sort_values(ascending=False)
    print(f"\n[{label}] NA% (top 15):")
    print(rep.head(15))

# -------------------------------------------------
# 1) Load backbone: spreads firm enriched
# -------------------------------------------------
SPREADS_ENR = INTERIM / "spreads_firm_enriched.parquet"
if not SPREADS_ENR.exists():
    raise FileNotFoundError(f"No encontré {SPREADS_ENR}")

panel = pd.read_parquet(SPREADS_ENR).copy()

# Normalización fecha (EOM)
panel["date"] = to_eom(panel["date"])

# Chequeo backbone
assert panel.duplicated(subset=["issuer", "date"]).sum() == 0, "Duplicados issuer-date en spreads backbone."
n_backbone = len(panel)

print("✅ Backbone (spreads) shape:", panel.shape)
print("✅ issuers:", panel["issuer"].nunique())
print("✅ rango:", panel["date"].min(), "→", panel["date"].max())

# -------------------------------------------------
# 2) Equity features: beta + ivol (ticker-date)
# -------------------------------------------------
EQ_FEAT = INTERIM / "equity_features_monthly.parquet"
equity_feat = pd.read_parquet(EQ_FEAT)
equity_feat["date"] = to_eom(equity_feat["date"])

# llave esperada
need_eq = {"ticker", "date"}
if not need_eq.issubset(equity_feat.columns):
    raise KeyError(f"Equity features debe tener {need_eq}. Tiene: {equity_feat.columns.tolist()}")

# dedup por ticker-date si hubiera
equity_feat = equity_feat.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")

# merge left por ticker-date (ticker viene del mapping del spread)
panel = panel.merge(equity_feat, on=["ticker", "date"], how="left")

# -------------------------------------------------
# 3) Fundamentals mensuales enriquecidos (ticker-date)
# -------------------------------------------------
FUND_M = INTERIM / "fundamentals_monthly_enriched.parquet"
fund = pd.read_parquet(FUND_M)
fund["date"] = to_eom(fund["date"])

need_f = {"ticker", "date"}
if not need_f.issubset(fund.columns):
    raise KeyError(f"Fundamentals mensuales debe tener {need_f}. Tiene: {fund.columns.tolist()}")

fund = fund.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")

# merge
panel = panel.merge(fund, on=["ticker", "date"], how="left", suffixes=("", "_fund"))

# -------------------------------------------------
# 4) Market power (market_share) (ticker-date)
# -------------------------------------------------
MP = INTERIM / "market_power_monthly.parquet"
mp = pd.read_parquet(MP)
mp["date"] = to_eom(mp["date"])

need_mp = {"ticker", "date", "market_share"}
if not need_mp.issubset(mp.columns):
    raise KeyError(f"Market power debe tener {need_mp}. Tiene: {mp.columns.tolist()}")

mp = mp.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")

panel = panel.merge(mp[["ticker", "date", "market_share"]], on=["ticker", "date"], how="left")

# -------------------------------------------------
# 5) Crédito (CRC alternativa iBoxx) (issuer-date)
# -------------------------------------------------
CRC = INTERIM / "crc_alt_iboxx_betas_monthly.parquet"
crc = pd.read_parquet(CRC)
crc["date"] = to_eom(crc["date"])

need_crc = {"issuer", "date", "crc_level_beta", "crc_slope_beta"}
if not need_crc.issubset(crc.columns):
    raise KeyError(f"CRC alternativa debe tener {need_crc}. Tiene: {crc.columns.tolist()}")

crc = crc.sort_values(["issuer", "date"]).drop_duplicates(["issuer", "date"], keep="last")

panel = panel.merge(
    crc[["issuer", "date", "crc_level_beta", "crc_slope_beta", "crc_r2", "crc_nobs"]],
    on=["issuer", "date"],
    how="left"
)

# -------------------------------------------------
# 6) Chequeos post-merge
# -------------------------------------------------
print("\n✅ Post-merge shape:", panel.shape)
print("Δ filas vs backbone:", len(panel) - n_backbone)
assert len(panel) == n_backbone, "El merge master cambió el número de filas (debería ser backbone)."

dup = panel.duplicated(subset=["issuer", "date"]).sum()
print("Duplicados issuer-date:", dup)
assert dup == 0, "Hay duplicados issuer-date tras merge master."

# NA report por bloques (ajustado a nombres típicos)
na_report(panel, ["ticker", "sector"], "Backbone mapping")
na_report(panel, ["beta", "ivol"], "Equity features")
na_report(panel, ["leverage", "log_assets", "cash_to_assets", "current_ratio", "roa"], "Fundamentals (típicos)")
na_report(panel, ["market_share"], "Market power")
na_report(panel, ["crc_level_beta", "crc_slope_beta", "crc_r2", "crc_nobs"], "Crédito (CRC alternativa)")

# -------------------------------------------------
# 7) Export pre-filtros / pre-transformaciones
# -------------------------------------------------
OUT_PRE = INTERIM / "panel_master_pre_filters.parquet"
panel.to_parquet(OUT_PRE, index=False)

print("\n✅ Exportado pre-panel en:", OUT_PRE)

## 14. Filtros de muestra (2015–2025) y consistencia del panel

Input:
- `data/intermediate/panel_master_pre_filters.parquet`

Output:
- `data/intermediate/panel_master_filtered.parquet`

Filtros:
- Rango temporal: 2015-01-31 a 2025-12-31 (fin de mes).
- Llave única: issuer-date.

Notas:
- No se agregan filtros “nuevos”. Solo se replica lo que ya estaba implícito (rango temporal) y chequeos.
- Si el notebook original hacía filtros adicionales (por NA, por cantidad mínima de obs, etc.), se agregan acá exactamente igual.

In [ ]:
# ==========================================================
# 14. Filtros de muestra y consistencia del panel
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------------------------
# Load pre-panel
# -------------------------------------------------
PRE = INTERIM / "panel_master_pre_filters.parquet"
if not PRE.exists():
    raise FileNotFoundError(f"No encontré {PRE}")

panel = pd.read_parquet(PRE).copy()

# Normalizar date a EOM (por seguridad)
panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")

# -------------------------------------------------
# Filtro temporal 2015–2025 (EOM)
# -------------------------------------------------
start = pd.Timestamp("2015-01-31")
end   = pd.Timestamp("2025-12-31")

n_before = len(panel)
panel = panel[(panel["date"] >= start) & (panel["date"] <= end)].copy()
n_after = len(panel)

print("✅ Filtro temporal aplicado")
print("Filas antes:", n_before)
print("Filas después:", n_after)
print("Rango:", panel["date"].min(), "→", panel["date"].max())

# -------------------------------------------------
# Chequeos de llaves
# -------------------------------------------------
required_keys = {"issuer", "date"}
missing = required_keys - set(panel.columns)
if missing:
    raise KeyError(f"Faltan columnas llave: {missing}")

dup = panel.duplicated(subset=["issuer", "date"]).sum()
print("Duplicados issuer-date:", dup)
assert dup == 0, "Hay duplicados issuer-date en panel filtrado."

# -------------------------------------------------
# (Opcional) filtros adicionales SOLO si ya existían en tu notebook
# - Por defecto NO aplico drop por NA, para no cambiar resultados.
# - Si querés replicar un drop específico, lo agregamos explícito acá.
# -------------------------------------------------
# Ejemplo (NO activo):
# panel = panel.dropna(subset=["spread_mean_bps", "beta", "ivol"]).copy()

# -------------------------------------------------
# Export filtered panel (intermedio)
# -------------------------------------------------
OUT = INTERIM / "panel_master_filtered.parquet"
panel.to_parquet(OUT, index=False)

print("\n✅ Exportado panel filtrado en:", OUT)
print("✅ n issuers:", panel["issuer"].nunique())

## 14.5 Merge macro/sector

Input:
- `data/intermediate/panel_master_filtered.parquet`
- `data/raw/market_sp500.parquet` (o .csv)
- `data/raw/iboxx_credit_indices_monthly.parquet`
- `data/intermediate/mkt_vol_monthly_60d.parquet`
- `data/intermediate/ivol_sector_monthly_60d.parquet`
- `data/raw/ticker_to_etf.csv` (o mapping equivalente GICS/TRBC → sector_ric)

Output:
- `data/intermediate/panel_master_with_macro.parquet`

Objetivo:
- Adjuntar shocks/series agregadas comunes y proxies sectoriales al panel firm-month:
  - `mkt_ret` mensual
  - `mkt_vol_60d` mensual
  - `credit_level`, `credit_slope` mensuales (iBoxx)
  - `ivol_sector` mensual vía ETF sectorial

Chequeos:
- Merge por `date` (macro) y por `ticker-date` (sector).
- No cambia N de filas respecto al panel filtrado.

In [ ]:
# ==========================================================
# 14.5 Merge macro/sector al panel filtrado
# ==========================================================

import pandas as pd
import numpy as np

def to_eom(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")

# -------------------------------------------------
# 1) Load panel filtrado
# -------------------------------------------------
PANEL_F = INTERIM / "panel_master_filtered.parquet"
panel = pd.read_parquet(PANEL_F).copy()
panel["date"] = to_eom(panel["date"])

n0 = len(panel)

# ==========================================================
# A) Shock de mercado: mkt_ret mensual (log-return)
# ==========================================================
mkt_parq = RAW / "market_sp500.parquet"
mkt_csv  = RAW / "market_sp500.csv"

if mkt_parq.exists():
    mkt = pd.read_parquet(mkt_parq)
elif mkt_csv.exists():
    mkt = pd.read_csv(mkt_csv)
else:
    raise FileNotFoundError("No encontré market_sp500 en RAW (parquet/csv).")

mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")

# Detectar precio o retorno
ret_candidates = ["mkt_ret", "ret", "return", "sp500_ret", "daily_ret"]
price_candidates = ["price", "close", "px_last", "index", "spx", "value"]

ret_col = next((c for c in ret_candidates if c in mkt.columns), None)
px_col  = next((c for c in price_candidates if c in mkt.columns), None)

if ret_col is not None:
    mkt["mkt_ret_d"] = pd.to_numeric(mkt[ret_col], errors="coerce")
elif px_col is not None:
    mkt["px"] = pd.to_numeric(mkt[px_col], errors="coerce")
    mkt = mkt.sort_values("date")
    mkt["mkt_ret_d"] = np.log(mkt["px"]).diff()
else:
    raise ValueError(f"market_sp500 no tiene retorno ni precio reconocible. Columnas: {list(mkt.columns)}")

mkt = mkt.dropna(subset=["mkt_ret_d"]).copy()
mkt["date_eom"] = mkt["date"].dt.to_period("M").dt.to_timestamp("M")

# Mensual: suma de log-returns diarios
mkt_m = (
    mkt.groupby("date_eom", as_index=False)["mkt_ret_d"]
       .sum()
       .rename(columns={"date_eom": "date", "mkt_ret_d": "mkt_ret"})
)

# Merge por date
panel = panel.merge(mkt_m, on="date", how="left")

# ==========================================================
# B) Volatilidad de mercado mkt_vol_60d (ya construida en Sección 9.2)
# ==========================================================
MV = INTERIM / "mkt_vol_monthly_60d.parquet"
if not MV.exists():
    raise FileNotFoundError(f"No encontré {MV} (corré Sección 9.2).")

mkt_vol = pd.read_parquet(MV).copy()
mkt_vol["date"] = to_eom(mkt_vol["date"])

# la columna puede venir como mkt_vol_60d o similar
mv_col = next((c for c in mkt_vol.columns if "mkt_vol" in c.lower()), None)
if mv_col is None:
    raise KeyError(f"No encontré columna mkt_vol en {MV}. Columnas: {list(mkt_vol.columns)}")

mkt_vol = mkt_vol.rename(columns={mv_col: "mkt_vol_60d"})[["date", "mkt_vol_60d"]].copy()

panel = panel.merge(mkt_vol, on="date", how="left")

# ==========================================================
# C) Shocks de crédito agregados: credit_level / credit_slope (iBoxx)
#     MISMA DEFINICIÓN que usaste en CRC alternativa:
#       credit_level = -Δlog(iboxx_ig_liquid)
#       credit_slope = (-Δlog(10y+)) - (-Δlog(1-5y))
# ==========================================================
IBOXX = RAW / "iboxx_credit_indices_monthly.parquet"
if not IBOXX.exists():
    raise FileNotFoundError(f"No encontré {IBOXX}")

iboxx = pd.read_parquet(IBOXX).copy()
iboxx["date"] = pd.to_datetime(iboxx["date"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")

# Wide por series
wide = (
    iboxx.pivot(index="date", columns="series", values="value")
         .sort_index()
)

need = ["iboxx_ig_liquid", "iboxx_overall_1_5y", "iboxx_overall_10y_plus"]
missing = [c for c in need if c not in wide.columns]
if missing:
    raise ValueError(f"Faltan series iBoxx requeridas: {missing}. Disponibles: {list(wide.columns)}")

ret = np.log(wide[need]).diff()

credit_level = -ret["iboxx_ig_liquid"]
credit_slope = (-ret["iboxx_overall_10y_plus"]) - (-ret["iboxx_overall_1_5y"])

credit = pd.DataFrame({
    "date": credit_level.index,
    "credit_level": credit_level.values,
    "credit_slope": credit_slope.values
}).dropna()

panel = panel.merge(credit, on="date", how="left")

# ==========================================================
# D) IVOL sectorial: mapear ticker → sector_ric y mergear ivol_sector (mensual)
# ==========================================================
IVS = INTERIM / "ivol_sector_monthly_60d.parquet"
if not IVS.exists():
    raise FileNotFoundError(f"No encontré {IVS} (corré Sección 9.1).")

ivol_sector = pd.read_parquet(IVS).copy()
ivol_sector["date"] = to_eom(ivol_sector["date"])

# la columna se llama ivol_sector
if "ivol_sector" not in ivol_sector.columns:
    # fallback por si vino con otro nombre
    cand = next((c for c in ivol_sector.columns if "ivol" in c.lower()), None)
    if cand is None:
        raise KeyError(f"No encontré ivol_sector en {IVS}. Columnas: {list(ivol_sector.columns)}")
    ivol_sector = ivol_sector.rename(columns={cand: "ivol_sector"})

# mapping ticker → sector_ric (de tu Sección 34 “Mapeo Sector TRBC - ETF”)
# Si lo tenés como parquet/csv en RAW, cargarlo.
MAP1 = RAW / "ticker_to_etf.parquet"
MAP2 = RAW / "ticker_to_etf.csv"

if MAP1.exists():
    ticker_to_etf = pd.read_parquet(MAP1)
elif MAP2.exists():
    ticker_to_etf = pd.read_csv(MAP2)
else:
    raise FileNotFoundError(
        "No encontré ticker_to_etf en RAW (ticker_to_etf.parquet/csv). "
        "Exportalo desde tu ETL o copiá el mapping que ya usabas en el notebook viejo."
    )

# Normalizar columnas esperadas
ticker_col = next((c for c in ticker_to_etf.columns if c.lower() == "ticker"), None)
sec_col    = next((c for c in ticker_to_etf.columns if "sector_ric" in c.lower() or c.lower() == "sector_ric"), None)

if ticker_col is None or sec_col is None:
    raise KeyError(
        f"ticker_to_etf debe tener columnas 'ticker' y 'sector_ric'. Tiene: {list(ticker_to_etf.columns)}"
    )

ticker_to_etf = ticker_to_etf.rename(columns={ticker_col: "ticker", sec_col: "sector_ric"}).copy()
ticker_to_etf["ticker"] = ticker_to_etf["ticker"].astype(str).str.upper().str.strip()
ticker_to_etf["sector_ric"] = ticker_to_etf["sector_ric"].astype(str).str.strip()

# Preparar panel ticker uppercase
if "ticker" not in panel.columns:
    raise KeyError("El panel filtrado no tiene columna 'ticker' (sale del mapping issuer→ticker).")

panel["ticker"] = panel["ticker"].astype(str).str.upper().str.strip()

panel = panel.merge(ticker_to_etf, on="ticker", how="left")

# Merge ivol_sector por sector_ric-date
ivol_sector_small = ivol_sector[["sector_ric", "date", "ivol_sector"]].drop_duplicates(["sector_ric", "date"])
panel = panel.merge(ivol_sector_small, on=["sector_ric", "date"], how="left")

# -------------------------------------------------
# Chequeos
# -------------------------------------------------
assert len(panel) == n0, "El merge macro/sector cambió el número de filas (debería ser LEFT desde panel)."

dup = panel.duplicated(["issuer", "date"]).sum()
assert dup == 0, f"Duplicados issuer-date tras merge macro/sector: {dup}"

print("✅ Merge macro/sector OK")
print("NA% mkt_ret:", panel["mkt_ret"].isna().mean())
print("NA% mkt_vol_60d:", panel["mkt_vol_60d"].isna().mean())
print("NA% credit_level:", panel["credit_level"].isna().mean())
print("NA% credit_slope:", panel["credit_slope"].isna().mean())
print("NA% ivol_sector:", panel["ivol_sector"].isna().mean())

# -------------------------------------------------
# Export intermedio
# -------------------------------------------------
OUT_MACRO = INTERIM / "panel_master_with_macro.parquet"
panel.to_parquet(OUT_MACRO, index=False)
print("✅ Exportado:", OUT_MACRO)

## 15. Transformaciones finales y winsorización (market_share)

Input:
- `data/intermediate/panel_master_filtered.parquet`
- `data/raw/fundamentals_with_market_share.csv`
- `data/clean/issuer_ticker_sector.parquet`

Output:
- `data/clean/panel_master.parquet`
- `data/clean/market_share_monthly.parquet` (si no existía o si se regenera)

Definiciones (idénticas al notebook original):
- `market_share` mensual: forward-fill entre reportes anuales
- `market_share = clip(0,1)`
- `market_share_w`: winsor 1%–99% (sobre toda la muestra de market_share)
- El log del spread NO se guarda acá (se construye en modelos con spread>0)

Chequeos:
- Merge por `issuer-date`
- NA% de market_share_w
- Distribución y cuantiles P1/P99

In [ ]:
# ==========================================================
# 15. Transformaciones / winsor (IDÉNTICO a tu notebook)
# - Construye market_share_monthly (ffill) desde raw anual
# - Clip market_share en [0,1]
# - market_share_w = winsor P1–P99
# - Merge a panel_master_filtered por issuer-date
# - Export final panel_master.parquet
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------------------------
# 1) Load panel filtrado (output de Sección 14)
# -------------------------------------------------
PANEL_IN = INTERIM / "panel_master_with_macro.parquet"
panel = pd.read_parquet(PANEL_IN).copy()
if not PANEL_IN.exists():
    raise FileNotFoundError(f"No encontré {PANEL_IN}")

panel = pd.read_parquet(PANEL_IN).copy()
panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")

# llaves mínimas
for c in ["issuer", "date"]:
    if c not in panel.columns:
        raise KeyError(f"panel_master_filtered debe tener columna '{c}'.")

# -------------------------------------------------
# 2) Construir (o cargar) market_share_monthly EXACTO
# -------------------------------------------------
MSHARE_M_FILE = CLEAN / "market_share_monthly.parquet"

if MSHARE_M_FILE.exists():
    ms_monthly = pd.read_parquet(MSHARE_M_FILE).copy()
    ms_monthly["date"] = pd.to_datetime(ms_monthly["date"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")
    print(f"✅ market_share_monthly ya existe: {MSHARE_M_FILE} | filas: {len(ms_monthly)}")

else:
    print("ℹ️ market_share_monthly no existe. Lo construyo desde fundamentals_with_market_share.csv (idéntico).")

    MSHARE_SRC = RAW / "fundamentals_with_market_share.csv"
    MAP_FILE   = CLEAN / "issuer_ticker_sector.parquet"

    if not MSHARE_SRC.exists():
        raise FileNotFoundError(f"No encontré {MSHARE_SRC}")
    if not MAP_FILE.exists():
        raise FileNotFoundError(f"No encontré {MAP_FILE}")

    mshare = pd.read_csv(MSHARE_SRC)
    mshare.columns = [c.strip().lower() for c in mshare.columns]

    for c in ["revenue", "industry_sales"]:
        if c in mshare.columns:
            mshare[c] = pd.to_numeric(mshare[c], errors="coerce")

    # market share a nivel fila
    if ("revenue" not in mshare.columns) or ("industry_sales" not in mshare.columns):
        raise KeyError("fundamentals_with_market_share.csv debe tener columnas 'revenue' e 'industry_sales'.")

    mshare["market_share"] = mshare["revenue"] / mshare["industry_sales"]

    # fecha = fin de mes del periodo reportado
    date_col = "period_end"
    if date_col not in mshare.columns:
        raise KeyError("fundamentals_with_market_share.csv debe tener columna 'period_end'.")

    mshare[date_col] = pd.to_datetime(mshare[date_col], errors="coerce")
    mshare["date"] = mshare[date_col].dt.to_period("M").dt.to_timestamp("M")

    # mapear issuer a partir del ticker base del RIC
    if "ric" not in mshare.columns:
        raise KeyError("fundamentals_with_market_share.csv debe tener columna 'ric'.")

    mapping = pd.read_parquet(MAP_FILE)[["issuer", "ticker", "sector"]].drop_duplicates()
    mshare["ric_base"] = mshare["ric"].astype(str).str.split(".").str[0].str.upper()
    mapping["ticker"]  = mapping["ticker"].astype(str).str.upper()

    ms = mshare.merge(mapping, left_on="ric_base", right_on="ticker", how="inner")

    # si hubiera múltiples filas por issuer/mes, promediamos esa “marca”
    ms_annual = (
        ms.groupby(["issuer", "date"], as_index=False)
          .agg({
              "market_share": "mean",
              "revenue": "sum",
              "industry_sales": "sum",
              "sector": "first"
          })
          .sort_values(["issuer", "date"])
    )

    # mensualizar por forward-fill entre reportes
    def monthly_ffill(df, issuer_col="issuer", date_col="date",
                      value_cols=("market_share",),
                      carry_along=("sector",)):
        out = []
        for iss, g in df.sort_values(date_col).groupby(issuer_col):
            if g.empty:
                continue
            grid = pd.DataFrame({
                issuer_col: iss,
                date_col: pd.period_range(g[date_col].min(), g[date_col].max(), freq="M").to_timestamp("M")
            })
            cols = [issuer_col, date_col] + list(value_cols) + list(carry_along)
            gg = grid.merge(g[cols], on=[issuer_col, date_col], how="left")

            for v in value_cols:
                gg[v] = gg[v].ffill()
            for c in carry_along:
                if c in gg.columns:
                    gg[c] = gg[c].ffill()

            out.append(gg)
        if not out:
            return pd.DataFrame(columns=[issuer_col, date_col, *value_cols, *carry_along])
        return pd.concat(out, ignore_index=True)

    ms_monthly = monthly_ffill(ms_annual, value_cols=("market_share",), carry_along=("sector",))

    # saneo en [0,1] + winsor 1%–99% (idéntico)
    ms_monthly["market_share"] = ms_monthly["market_share"].clip(0, 1)
    q1, q99 = ms_monthly["market_share"].quantile([0.01, 0.99])
    ms_monthly["market_share_w"] = ms_monthly["market_share"].clip(q1, q99)

    # guardar
    ms_monthly.to_parquet(MSHARE_M_FILE, index=False)
    print(f"✅ market_share mensual guardado en {MSHARE_M_FILE}, filas: {len(ms_monthly)}")

# normalizar columnas mínimas
need_ms = {"issuer", "date", "market_share", "market_share_w"}
miss = need_ms - set(ms_monthly.columns)
if miss:
    raise KeyError(f"market_share_monthly debe tener {need_ms}. Faltan: {miss}")

ms_monthly = ms_monthly.copy()
ms_monthly["date"] = pd.to_datetime(ms_monthly["date"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")
ms_monthly = ms_monthly.sort_values(["issuer", "date"]).drop_duplicates(["issuer", "date"], keep="last")

# -------------------------------------------------
# 3) Merge market share al panel (issuer-date, LEFT)
# -------------------------------------------------
n_before = len(panel)

# si en el panel existía un market_share previo, lo pisamos con el “oficial”
for c in ["market_share", "market_share_w"]:
    if c in panel.columns:
        panel = panel.drop(columns=[c])

panel = panel.merge(
    ms_monthly[["issuer", "date", "market_share", "market_share_w"]],
    on=["issuer", "date"],
    how="left"
)

assert len(panel) == n_before, "El merge de market share cambió filas (debería ser LEFT)."

# -------------------------------------------------
# 4) QA (idéntico espíritu al original)
# -------------------------------------------------
print("\n=== QA market_share ===")
print("market_share NA%:", panel["market_share"].isna().mean())
print("market_share_w NA%:", panel["market_share_w"].isna().mean())

s = pd.to_numeric(panel["market_share"], errors="coerce")
desc = s.dropna().describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99])
print("\nmarket_share describe:")
print(desc)

if not s.dropna().empty:
    p1, p99 = s.quantile([0.01, 0.99])
    outliers = ((s < p1) | (s > p99)).sum()
    print(f"\nP1={p1:.6f} | P99={p99:.6f} | outliers(<P1 o >P99)={int(outliers)}")

# -------------------------------------------------
# 5) Export final: panel_master.parquet
# -------------------------------------------------
OUT_FINAL = CLEAN / "panel_master.parquet"
panel.to_parquet(OUT_FINAL, index=False)

print("\n✅ Panel FINAL guardado en:", OUT_FINAL)
print("Filas:", len(panel), "| issuers:", panel["issuer"].nunique())
print("Rango:", panel["date"].min(), "→", panel["date"].max())